## Why this exists

Start with the engineering problem. Know what would go wrong if this mechanism did not exist.


## Capability contract

**Capability:** C03 — Correctness Contracts & Idempotency  
**Workstream:** Revenue Data Platform  
**Primary roles:** Data Engineer|Analytics Engineer|Platform Engineer  
**You will prove:** Detect a correctness failure


## 🖊️ Sketch note

![Sketch note — Mini Analytical Platform](../assets/sketch-notes/14-mini-analytical-platform.svg)

> **Visual-first rule:** spend 30–60 seconds explaining the sketch in your own words before reading the implementation. Then return to it after the experiment and redraw it from memory.


## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S03 — Data quality (DATA-1063)** → [`../work_simulations/03_acme_commerce_quality_gate/README.md`](../work_simulations/03_acme_commerce_quality_gate/README.md)
- **S04 — Incremental and idempotent pipelines (DATA-1070)** → [`../work_simulations/04_acme_commerce_duplicate_rerun/README.md`](../work_simulations/04_acme_commerce_duplicate_rerun/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


## Measure

Record an observable result that lets you compare your prediction with what actually happened.


## Learning objectives

- connect ingestion, quality and curation;
- identify system boundaries;
- record run-level evidence;
- explain one failure across multiple stages.


## 1. Architecture

```text
source fixture/API
       ↓
   raw JSONL  ─────────────┐
       ↓                   │
 validation + quality      │ run metadata
       ↓                   │
 curated Parquet           │
       ↓                   │
 DuckDB analytical view ←──┘
       ↓
 revenue / customer data product
```

The platform has two kinds of state:
- **data state**: raw and curated records;
- **operational state**: run ID, row counts, quality result and status.

## 2. Predict before you run

> If the quality gate fails, should the analytical view be refreshed with the new data anyway? What evidence would an operator need to explain the decision?

In [ ]:
import importlib.util, subprocess, sys

REQUIRED = {"duckdb": "duckdb>=1.0,<2", "pyarrow": "pyarrow>=16,<22", "pandas": "pandas>=2,<4", "requests": "requests>=2.31,<3"}
missing = [spec for module, spec in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json, uuid
import pandas as pd
import duckdb

ROOT = Path("data/14_mini_platform")
ROOT.mkdir(parents=True, exist_ok=True)
RAW = ROOT / "raw_orders.jsonl"
CURATED = ROOT / "curated_orders.parquet"
DB = ROOT / "platform.duckdb"
if DB.exists(): DB.unlink()

source_records = [
    {"order_id":"O1", "customer_id":"C1", "amount":120.0, "status":"paid", "event_time":"2026-09-30T04:00:00+00:00"},
    {"order_id":"O2", "customer_id":"C2", "amount":80.0, "status":"paid", "event_time":"2026-09-30T04:02:00+00:00"},
    {"order_id":"O3", "customer_id":"C1", "amount":50.0, "status":"pending", "event_time":"2026-09-30T04:04:00+00:00"},
    {"order_id":"O4", "customer_id":"C3", "amount":40.0, "status":"paid", "event_time":"2026-09-30T04:06:00+00:00"},
]
RAW.write_text("\n".join(json.dumps(r) for r in source_records) + "\n", encoding="utf-8")
run_id = str(uuid.uuid4())
started = datetime.now(timezone.utc)
print("Run:", run_id)

## 3. Ingest and normalize

In [ ]:
raw = [json.loads(line) for line in RAW.read_text(encoding="utf-8").splitlines() if line.strip()]
orders = pd.DataFrame(raw)
orders["event_time"] = pd.to_datetime(orders["event_time"], utc=True)
orders["amount"] = orders["amount"].astype(float)
print("Raw rows:", len(raw))
display(orders)

## 4. Quality gate

In [ ]:
checks = []
checks.append({"check":"not_null:order_id", "passed":orders["order_id"].notna().all(), "observed":int(orders["order_id"].isna().sum())})
checks.append({"check":"unique:order_id", "passed":orders["order_id"].is_unique, "observed":int(orders["order_id"].duplicated().sum())})
checks.append({"check":"positive:amount", "passed":(orders["amount"] > 0).all(), "observed":int((orders["amount"] <= 0).sum())})
checks.append({"check":"allowed:status", "passed":orders["status"].isin({"pending","paid","cancelled"}).all(), "observed":int((~orders["status"].isin({"pending","paid","cancelled"})).sum())})
quality = pd.DataFrame(checks)
quality["run_id"] = run_id
display(quality)
quality_ok = bool(quality["passed"].all())
assert quality_ok, "Quality gate failed; curated publication must stop."

## 5. Publish curated data and build the analytical product

In [ ]:
orders.to_parquet(CURATED, index=False)
con = duckdb.connect(str(DB))
con.execute(f"CREATE OR REPLACE VIEW curated_orders AS SELECT * FROM read_parquet('{CURATED.as_posix()}')")

customer_revenue = con.execute("""
    SELECT customer_id,
           COUNT(*) AS orders,
           SUM(CASE WHEN status = 'paid' THEN amount ELSE 0 END) AS paid_revenue
    FROM curated_orders
    GROUP BY customer_id
    ORDER BY paid_revenue DESC
""").fetchdf()
display(customer_revenue)
assert customer_revenue["paid_revenue"].sum() == 240.0

## 6. Measure

The platform should expose a few simple operational measures before adding a monitoring system.

In [ ]:
print("Input rows:", len(raw))
print("Curated rows:", len(orders))
print("Quality checks:", len(quality))
print("Quality failures:", int((~quality["passed"]).sum()))
print("Paid revenue:", float(customer_revenue["paid_revenue"].sum()))
assert len(raw) == len(orders) == 4
assert int((~quality["passed"]).sum()) == 0

## 6. Observability: record the run, not just the data

A useful first layer of observability answers: what ran, when, how much data moved, whether quality passed, and what artifact was produced.

In [ ]:
finished = datetime.now(timezone.utc)
run = pd.DataFrame([{
    "run_id": run_id,
    "started_at": started,
    "finished_at": finished,
    "status": "success",
    "input_rows": len(raw),
    "curated_rows": len(orders),
    "quality_checks": len(quality),
    "quality_failed": int((~quality["passed"]).sum()),
    "raw_bytes": RAW.stat().st_size,
    "curated_bytes": CURATED.stat().st_size,
}])
run.to_json(ROOT / "run_metadata.json", orient="records", indent=2, date_format="iso")
display(run)
print("Artifacts:")
for p in [RAW, CURATED, ROOT / "run_metadata.json", DB]:
    print(" -", p)

## 💥 7. Break the platform

Introduce a duplicate order. The correct behavior is to stop before replacing the curated artifact or refreshing the analytical product.

In [ ]:
broken = orders.copy()
broken.loc[len(broken)] = broken.iloc[0]
broken_checks = [
    {"check":"unique:order_id", "passed":broken["order_id"].is_unique, "observed":int(broken["order_id"].duplicated().sum())},
    {"check":"positive:amount", "passed":(broken["amount"] > 0).all(), "observed":int((broken["amount"] <= 0).sum())},
]
broken_quality = pd.DataFrame(broken_checks)
display(broken_quality)
assert not broken_quality["passed"].all()
print("Decision: BLOCK curated publication; keep the last known good artifact.")

## 8. Diagnose → Improve

The platform now has a useful separation of concerns:

- raw data is preserved for replay;
- quality evidence explains why publication was blocked;
- curated data changes only after the gate passes;
- analytical views consume the curated contract;
- run metadata tells an operator what happened.

This is intentionally small. The point is to make the dependency chain visible before adding orchestration or distributed infrastructure.

## 9. Production bridge

At 1,000× volume, move durable artifacts to object storage, partition data by workload-relevant columns, and use an execution engine appropriate to scale. Add orchestration, retries, lineage, alerting, access control, schema evolution and retention.

If the dependency fails, the run should be retryable without corrupting the last known good dataset. If a backfill is required, it should be distinguishable from a normal scheduled run.

**Architectural principle:** make the data product trustworthy before making the platform large.

## 10. Explain it in 60–90 seconds

> Walk through one order from source to analytical result. Where can it fail, what evidence remains, and what prevents a failed run from silently replacing trusted data?

## 11. Challenge — architecture review

Design the next version for 100 million orders/day with:
- <15 minute freshness;
- replayable ingestion;
- quality checks before publication;
- customer-level analytics;
- auditable runs.

State your assumptions, bottlenecks, failure modes, recovery strategy, observability and the first technology you would replace as scale grows.

## Takeaways

- A platform is a connected set of contracts, state and operational evidence.
- Quality belongs in the publication path, not only in a dashboard.
- Run metadata makes pipeline behavior observable and recoverable.
- The capstone now has a concrete source → raw → quality → curated → analytics path.
- The next curriculum layer can add orchestration, lineage and richer observability without changing the mental model.